<a href="https://colab.research.google.com/github/Charan-BE21B010/My-Projects/blob/my-projects/Electricity_Forecasting_Demand.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Overview
- You will perform electricity demand forecasting using the data provided in a Google Colaboratory notebook.
- For instructions on using Google Colaboratory, please refer to this [link](https://colab.research.google.com/?hl=ja#scrollTo=Wf5KrEb6vrkR).
- The provided data consists of two types:
  - electricity_demand.csv: Electricity demand values within the Kansai Electric Power (JP) jurisdiction.
  - {location}.csv: Actual weather data for specific locations.

## Rules
- The forecasting target period is January 1, 2023, 00:00:00 – December 31, 2023, 23:00:00.
- Electricity demand values can only be used up until 23:00:00 of the day before the target date.
- There are no restrictions on forecasting methods (choose methods available in the free-tier plan).
- The use of additional data from libraries such as jpholiday is permitted.
  - If additional libraries are used, include the installation command in the notebook (e.g., ! pip install pandas).

## Required Deliverables
Use both code and Markdown as needed to provide the following information:

- Model Development for Electricity Demand Forecasting
  - Justify the choice of method and, if applicable, describe any hypotheses regarding feature engineering.
- Evaluation of Forecasting Performance Using Actual Data
  - Specify and justify the selected evaluation metrics.
- Error Analysis and Model Challenges
- Discuss issues identified through error analysis.
- Hypotheses for Accuracy Improvement
  - Provide potential strategies for enhancing model performance.
- Expected Benefits of Model Deployment
  - Explain the potential impact and advantages of implementing the model.

## Submission
- Once coding and execution are completed in Google Colaboratory, download the executed notebook and submit it.
- Submit your notebook through the designated agent.

In [127]:
import pandas as pd
import numpy as np
demand = pd.read_csv("/content/demand.csv")
hikone = pd.read_csv("/content/hikone.csv")

In [128]:
# Attempt to convert 'datetime' column to datetime type, allowing for mixed formats
demand['datetime'] = pd.to_datetime(demand['datetime'], errors='coerce', dayfirst=False)
hikone['datetime'] = pd.to_datetime(hikone['datetime'], errors='coerce', dayfirst=False)

In [129]:
# Define the start and end date
start_date = '2023-01-01 00:00:00'
end_date = '2023-12-31 23:00:00'

# Filter data within the date range
filtered_demand = demand[(demand['datetime'] >= start_date) & (demand['datetime'] <= end_date)]
filtered_hikone = hikone[(hikone['datetime'] >= start_date) & (hikone['datetime'] <= end_date)]

# Optionally, merge the filtered data
data = pd.merge(filtered_demand, filtered_hikone, on='datetime', how='inner')


In [130]:
data.shape

(3456, 9)

In [131]:
data.describe()

,datetime,actual_performance(10000 kW),precipitation,temperature,dew_point_temperature,humidity,wind_speed,snowfall
count,3456,3456.000000,3456.000000,3456.000000,3456.000000,3456.000000,3456.000000,3456.0
mean,2023-06-22 11:29:59.999999744,1563.509259,0.175781,16.508623,11.795631,75.161169,2.751128,0.0
min,2023-01-01 00:00:00,955.000000,0.000000,-2.500000,-4.900000,28.000000,0.000000,0.0
25%,2023-03-27 05:45:00,1315.000000,0.000000,8.700000,4.000000,65.000000,1.400000,0.0
50%,2023-06-21 23:30:00,1508.000000,0.000000,16.700000,11.100000,76.000000,2.100000,0.0
75%,2023-09-17 11:15:00,1741.000000,0.000000,23.700000,20.500000,87.000000,3.500000,0.0
max,2023-12-12 23:00:00,2633.000000,21.000000,36.400000,26.300000,100.000000,11.800000,0.0
std,NaN,327.599428,0.914027,8.797893,8.445358,14.329139,1.979453,0.0


In [132]:
data.isnull().count()

,0
datetime,3456
actual_performance(10000 kW),3456
precipitation,3456
temperature,3456
dew_point_temperature,3456
humidity,3456
wind_speed,3456
wind_direction,3456
snowfall,3456


In [139]:
# Convert 'datetime' to datetime object and extract useful features
data['datetime'] = pd.to_datetime(data['datetime'])
data['hour'] = data['datetime'].dt.hour
data['day'] = data['datetime'].dt.day
data['month'] = data['datetime'].dt.month

# Drop the original 'datetime' column
data.drop(columns=['datetime'], inplace=True)

# Define features and target
X = data.drop(columns=['temperature'])
y = data['temperature']

In [145]:
# Split the data into training and testing sets
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [141]:
# Preprocessing: Handle categorical and numerical features
numerical_features = ['precipitation', 'dew_point_temperature', 'humidity', 'wind_speed', 'snowfall', 'hour', 'day', 'month']
categorical_features = ['wind_direction']

In [136]:
# Create a column transformer for preprocessing
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(), categorical_features)
    ])

In [137]:
# Define the models
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42)
}


In [149]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
# Train and evaluate each model
results = {}
for model_name, model in models.items():
    # Create a pipeline with preprocessing and the model
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    # Train the model
    pipeline.fit(X_train, y_train)

    # Make predictions
    y_pred = pipeline.predict(X_test)

    # Evaluate the model
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    results[model_name] = {'MAE': mae, 'RMSE': rmse}


In [144]:
# Display the results
for model_name, metrics in results.items():
    print(f"{model_name}:")
    print(f"  MAE: {metrics['MAE']}")
    print(f"  RMSE: {metrics['RMSE']}")

Linear Regression:
  MAE: 0.31907859241734604
  RMSE: 0.4281519862654546
Random Forest:
  MAE: 0.16529046242774637
  RMSE: 0.2409150996930579
Gradient Boosting:
  MAE: 0.2515332741996388
  RMSE: 0.3322378844643022
